# FRED Time Series — MCP & TimeSeriesDataFetcherAgent Tests

Tests the MCP interface directly and via the `TimeSeriesDataFetcherAgent`.

In [1]:
import os
import sys
import json

sys.path.insert(0, os.path.abspath('../..'))

from apps.agentic.core.utils import set_anthropic_env
from langchain_mcp_adapters.client import MultiServerMCPClient

set_anthropic_env(filedir='../../.keys')

## 1. Direct MCP Tool Call

Verify the MCP server is reachable and `fred.series_observations` returns expected data.

In [3]:
from lib.mcp_client import MCPClient, MCPClientConfig
from apps.agentic.core.constants import MCP_URL

mcp_config = MCPClientConfig(url=MCP_URL)

async def call_tool(tool_name, arguments=None):
    async with MCPClient(mcp_config) as client:
        return await client.call_tool(tool_name, arguments or {})

async def list_mcp_tools():
    async with MCPClient(mcp_config) as client:
        return await client.list_tools()

tools = await list_mcp_tools()
for t in tools:
    print(f'{t.name}: {t.description}')

fred_category_children: List the child categories for a FRED category.
fred_category_series: List the series contained within a FRED category.
fred_series_info: Fetch metadata for a single FRED series.
fred_series_observations: Return observations for a series (limit defaults to 100).
fred_series_updates: Return recently updated FRED series.
list_releases: List the series that belong to a FRED release.
fred_release_series: List the series that belong to a FRED release.
tiingo_series_info: Fetch metadata for a Tiingo ticker (ETF, mutual fund, or stock), including its available date range.
tiingo_price_series: Return the end-of-day price series (OHLCV plus split/dividend-adjusted prices) for a Tiingo ticker. Provide start_date/end_date as YYYY-MM-DD for a range; omit for the latest day.


In [4]:
SERIES_ID = 'BOGMBBM'

result = await call_tool('fred_series_observations', {'series_id': SERIES_ID, 'limit': 5})

print(result)
if result.structuredContent:
    data = result.structuredContent['result']
    print(f"count  : {data['count']}")
    print(f"limit  : {data['limit']}")
    print(f"offset : {data['offset']}")
    print()
    for obs in data['observations']:
        print(obs)

meta=None content=[TextContent(type='text', text='{\n  "realtime_start": "2026-06-21",\n  "realtime_end": "2026-06-21",\n  "order_by": "observation_date",\n  "sort_order": "asc",\n  "count": 808,\n  "offset": 0,\n  "limit": 5,\n  "observations": [\n    {\n      "realtime_start": "2026-06-21",\n      "realtime_end": "2026-06-21",\n      "date": "1959-01-01",\n      "value": "18.9"\n    },\n    {\n      "realtime_start": "2026-06-21",\n      "realtime_end": "2026-06-21",\n      "date": "1959-02-01",\n      "value": "18.6"\n    },\n    {\n      "realtime_start": "2026-06-21",\n      "realtime_end": "2026-06-21",\n      "date": "1959-03-01",\n      "value": "18.4"\n    },\n    {\n      "realtime_start": "2026-06-21",\n      "realtime_end": "2026-06-21",\n      "date": "1959-04-01",\n      "value": "18.7"\n    },\n    {\n      "realtime_start": "2026-06-21",\n      "realtime_end": "2026-06-21",\n      "date": "1959-05-01",\n      "value": "18.6"\n    }\n  ]\n}', annotations=None, meta=None)

In [5]:
from datetime import datetime

result = await call_tool('fred_series_observations', {'series_id': SERIES_ID, 'limit': 10000})

if result.structuredContent:
    observations = result.structuredContent['result']['observations']

    timestamps = []
    values = []
    for obs in observations:
        if obs['value'] == '.':
            continue
        timestamps.append(datetime.strptime(obs['date'], '%Y-%m-%d'))
        values.append(float(obs['value']))

    print(f'Series  : {SERIES_ID}')
    print(f'Points  : {len(timestamps)}')
    print(f'Start   : {timestamps[0].date()}')
    print(f'End     : {timestamps[-1].date()}')
    print(f'Min val : {min(values):.4f}')
    print(f'Max val : {max(values):.4f}')
    print(f'Number of Observations: {len(observations)}')

Series  : BOGMBBM
Points  : 808
Start   : 1959-01-01
End     : 2026-04-01
Min val : 12.3000
Max val : 4193.2000
Number of Observations: 808


## 2. MCP Tool Discovery via `MultiServerMCPClient`

Verify that `langchain-mcp-adapters` discovers tools from the MCP server correctly.

In [7]:
from typing import Any

connections: dict[str, Any] = {'fred': {'transport': 'sse', 'url': MCP_URL}}
client = MultiServerMCPClient(connections)
discovered_tools = await client.get_tools()

print(f'Discovered {len(discovered_tools)} tools:')
for t in discovered_tools:
    print(f'  {t.name}: {t.description}')

Discovered 9 tools:
  fred_category_children: List the child categories for a FRED category.
  fred_category_series: List the series contained within a FRED category.
  fred_series_info: Fetch metadata for a single FRED series.
  fred_series_observations: Return observations for a series (limit defaults to 100).
  fred_series_updates: Return recently updated FRED series.
  list_releases: List the series that belong to a FRED release.
  fred_release_series: List the series that belong to a FRED release.
  tiingo_series_info: Fetch metadata for a Tiingo ticker (ETF, mutual fund, or stock), including its available date range.
  tiingo_price_series: Return the end-of-day price series (OHLCV plus split/dividend-adjusted prices) for a Tiingo ticker. Provide start_date/end_date as YYYY-MM-DD for a range; omit for the latest day.


In [8]:
from apps.agentic.core.mcp_tool_registry import MCPToolRegistry
from apps.agentic.core.constants import MCP_URL

await MCPToolRegistry.initialize(
    servers={"fred": {"transport": "sse", "url": MCP_URL}},
    required=["fred_series_observations"],
)

DEBUG:    MCPToolRegistry discovered 9 tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'list_releases', 'fred_release_series', 'tiingo_series_info', 'tiingo_price_series']
INFO:     MCPToolRegistry initialized with 9 tools: ['fred_category_children', 'fred_category_series', 'fred_series_info', 'fred_series_observations', 'fred_series_updates', 'list_releases', 'fred_release_series', 'tiingo_series_info', 'tiingo_price_series']


## 3. TimeSeriesDataFetcherAgent — Direct Invocation

Create the agent via `TimeSeriesDataFetcherAgent.create()` and invoke it directly. The agent wraps the MCP tools with caching, so it stores the full series in `SeriesCache` and returns a compact `SeriesRef` JSON (not the raw observations).

In [9]:
from langchain_core.messages import HumanMessage
from langchain_core.runnables import RunnableConfig
import shortuuid

from apps.agentic.db.series_cache import SeriesCache
from apps.agentic.agents.time_series.time_series_data_fetcher_agent import TimeSeriesDataFetcherAgent

SeriesCache.initialize()

agent = await TimeSeriesDataFetcherAgent.create()
print(f'Agent initialized with tools: {list(agent.tool_node.tools_by_name.keys())}')

INFO:     SeriesCache initialized: postgresql://yada@localhost/yada
DEBUG:    TimeSeriesDataFetcherAgent using MCP tools: ['fred_series_observations', 'fred_series_info', 'tiingo_price_series', 'tiingo_series_info']
DEBUG:    TimeSeriesDataFetcherAgent prompt: 
            <instructions>
            You are a time series data fetcher agent. Use the available MCP tools to retrieve time series
            data from external data sources.

            The tools return a SeriesRef JSON object — pass it back unmodified as your response.
            Do not summarize, reformat, or interpret it.

            When fetching FRED data use the fred_series_observations tool with a limit of
            10000 to retrieve the full series.

            For ETF, mutual fund, or stock price history, use the tiingo_price_series tool
            with the ticker symbol (e.g. SPY); it returns the full adjusted-close price series.
            </instructions>
            
Agent initialized with tools: ['fred_s

In [10]:
state = {'messages': [HumanMessage(content=f'Fetch the {SERIES_ID} series from FRED with a limit of 10000.')]}
run_config = RunnableConfig(configurable={'thread_id': shortuuid.uuid()})

result = await agent.agent.ainvoke(state, run_config)
response = result['messages'][-1].content
print('Agent response (SeriesRef):', response)

INFO:     [TimeSeriesDataFetcherAgent] routing → fred_series_observations({'series_id': 'BOGMBBM', 'limit': 10000})
DEBUG:    fred_series_observations: cache miss for fred:BOGMBBM — fetching
DEBUG:    SeriesCache: stored fred:BOGMBBM → 287fb4a1-23cd-4d8a-861e-18e9a9a73d15
DEBUG:    fred_series_observations: cached fred:BOGMBBM → 287fb4a1-23cd-4d8a-861e-18e9a9a73d15
Agent response (SeriesRef): {"source": "fred", "native_id": "BOGMBBM", "cache_id": "287fb4a1-23cd-4d8a-861e-18e9a9a73d15"}
